In [1]:
import json
import duckdb
import requests

In [2]:
db_connection = duckdb.connect('loadsmart/dev.duckdb')

In [3]:
manifest = 'loadsmart/target/manifest.json'
with open(manifest, 'r', encoding='utf-8') as f:
    manifest = json.load(f)

In [4]:
table_content = []
for item_id, item in manifest.get('nodes', {}).items():
    if item.get('resource_type') == 'model':
        table = item.get('name')
        table_ds = item.get('description', '')
        columns = item.get('columns', {})
        columns_ds = [f" - {col_name}: {col_info.get('description', '')}" for col_name, col_info in columns.items()]
        table_content.append(f"Table: {table}\nDescription: {table_ds}\nColumns:\n" + "\n".join(columns_ds))

In [5]:
ai_guide = "\n\n".join(table_content)

In [6]:
def ask_ai_db_question(question):
    ai_prompt = f"""
        You are an expert DuckDB SQL AI.

        SCHEMA METADATA (Generated from dbt):
        {ai_guide}

        CRITICAL BEHAVIORAL RULES:
        1. You MUST read and strictly obey all `description` fields in the schema metadata above. They contain mandatory rules for handling historical dates, avoiding CURRENT_DATE, and filtering metrics.
        2. Write a valid DuckDB SQL query to answer the question.
        3. Return ONLY the executable SQL query in a markdown code block (```sql ... ```).

        Question: {question}
    """

    api_response = requests.post(
        "http://localhost:11434/api/generate",
        json = {
            'model': 'qwen2.5-coder:7b',
            'prompt': ai_prompt,
            'stream': False,
            'options': {'temperature': 0.0}
        }
    )

    llm_response = api_response.json()
    raw_sql = llm_response.get('response', '')
    cleaned_sql = raw_sql.replace('```sql', '').replace('```', '').strip()

    print(f"--- SQL ---\n{cleaned_sql}\n ---")

    # old code (no retry, crashes if the SQL is broken):
    # result_table = db_connection.execute(cleaned_sql).fetchdf()
    # return cleaned_sql, result_table

    # new code: if the SQL fails, send the error back to the model and try once more
    try:
        result_table = db_connection.execute(cleaned_sql).fetchdf()
    except Exception as e:
        fix_prompt = ai_prompt + f"\n\nThat query failed with this error:\n{e}\n\nFix it and return only the corrected SQL."
        api_response = requests.post(
            "http://localhost:11434/api/generate",
            json = {
                'model': 'qwen2.5-coder:7b',
                'prompt': fix_prompt,
                'stream': False,
                'options': {'temperature': 0.0}
            }
        )
        cleaned_sql = api_response.json().get('response', '').replace('```sql', '').replace('```', '').strip()
        print(f"--- retry SQL ---\n{cleaned_sql}\n ---")
        result_table = db_connection.execute(cleaned_sql).fetchdf()

    return cleaned_sql, result_table

In [ ]:
questions = [
    "How many loads were delivered in the last full month available in the data?",
    "Which shipper had the highest total book price?",
    "What is the average book price per load by pickup state?",
    "What are the top 5 lanes by number of delivered loads?",
    "Which carrier moved the most loads into Texas?",
    "How does the average book price compare between intrastate and interstate loads?",
    "For the shipper with the most delivered loads, how did monthly volume change across the period covered by the data?",
    "Among lanes with at least 10 delivered loads, which had the highest average book price?",
]

for question in questions:
    print(f"QUESTION: {question}")
    try:
        sql_query, result_df = ask_ai_db_question(question)
        display(result_df)
    except Exception as e:
        print(f"FAILED: {e}")
    print()

Q: How many loads were delivered in the last full month available in the data?


--- SQL ---
SELECT COUNT(loadsmart_id) AS delivered_loads
FROM fact_loadsmart
WHERE load_was_cancelled = FALSE
  AND delivery_date >= DATE '2024-12-01'
  AND delivery_date < DATE '2025-01-01';
 ---


,delivered_loads
0,442



Q: Which shipper had the highest total book price?


--- SQL ---
SELECT 
    s.shipper_name, 
    SUM(f.book_price) AS total_book_price
FROM 
    dim_shippers s
JOIN 
    fact_loadsmart f ON s.shipper_key = f.shipper_key
GROUP BY 
    s.shipper_name
ORDER BY 
    total_book_price DESC
LIMIT 1;
 ---


...

--- SQL ---
WITH delivered_loads AS (
    SELECT
        shipper_key,
        DATE_TRUNC('month', delivery_date) AS month,
        COUNT(*) AS load_count
    FROM
        fact_loadsmart
    WHERE
        load_was_cancelled = FALSE
    GROUP BY
        shipper_key,
        month
),
shipper_load_counts AS (
    SELECT
        shipper_key,
        COUNT(*) AS total_loads
    FROM
        fact_loadsmart
    WHERE
        load_was_cancelled = FALSE
    GROUP BY
        shipper_key
),
shipper_with_most_loads AS (
    SELECT
        shipper_key
    FROM
        shipper_load_counts
    ORDER BY
        total_loads DESC
    LIMIT 1
),
monthly_volume_change AS (
    SELECT
        month,
        load_count,
        LAG(load_count) OVER (ORDER BY month) AS prev_month_load_count
    FROM
        delivered_loads
    WHERE
        shipper_key = (SELECT shipper_key FROM shipper_with_most_loads)
)
SELECT
    month,
    load_count,
    prev_month_load_count,
    (load_count - prev_month_load_count) AS 

,month,load_count,prev_month_load_count,volume_change
0,2024-01-01,127,<NA>,<NA>
1,2024-02-01,108,127,-19
2,2024-03-01,178,108,70
3,2024-04-01,191,178,13
4,2024-05-01,163,191,-28
5,2024-06-01,165,163,2
6,2024-07-01,137,165,-28
7,2024-08-01,126,137,-11
8,2024-09-01,137,126,11
9,2024-10-01,153,137,16



Q: Among lanes with at least 10 delivered loads, which had the highest average book price?


--- SQL ---
WITH delivered_loads AS (
    SELECT
        lane_key,
        AVG(book_price) AS avg_book_price
    FROM
        fact_loadsmart
    WHERE
        load_was_cancelled = FALSE
    GROUP BY
        lane_key
    HAVING
        COUNT(loadsmart_id) >= 10
)
SELECT
    d.lane_key,
    d.avg_book_price,
    l.lane
FROM
    delivered_loads d
JOIN
    dim_lanes l ON d.lane_key = l.lane_key
ORDER BY
    d.avg_book_price DESC
LIMIT 1;
 ---


,lane_key,avg_book_price,lane
0,d8942e8a28f5eea4a55f5bda2a2a8f85,6800.0,"Stockton,CA -> Parrish,FL"


## Iteration log

First run used qwen2.5-coder:1.5b and failed 6 of 8 questions - crashed on invalid SQL, made up columns that don't exist, one got stuck repeating itself and never finished. Switched to qwen2.5-coder:7b, which fixed most of it.

Three things still needed fixing after that:
- Q1 used today's date instead of the data's date range. Fixed with a note on `delivery_date` in `schema.yml` saying it's historical data.
- Q7 returned one month instead of the full trend. Ended up fixing itself once other columns were better documented, no rule needed.
- Q8 crashed on a typo in its own SQL. Fixed by making it retry once when a query fails (see `ask_ai_db_question` above).

One assumption: Q1 asks for "the last full month available in the data." I read that as December 2024, since the data barely has anything after that. That's written into `delivery_date`'s description in `schema.yml`.

## Two questions a stakeholder would actually ask

- **How many loads were cancelled?** Every cancellation is lost time and lost revenue, so this is a basic number ops would want to track.
- **What is the total book price across all loads?** Basically total revenue tracking. Probably one of the first number anyone in the business would ask for.

In [8]:
stakeholder_questions = [
    "How many loads were cancelled?",
    "What is the total book price across all loads?",
]

for question in stakeholder_questions:
    print(f"Q: {question}")
    sql_query, result_df = ask_ai_db_question(question)
    display(result_df)
    print()

Q: How many loads were cancelled?


--- SQL ---
SELECT COUNT(*) AS cancelled_loads
FROM fact_loadsmart
WHERE load_was_cancelled = TRUE;
 ---


,cancelled_loads
0,516



Q: What is the total book price across all loads?


--- SQL ---
SELECT SUM(book_price) AS total_book_price
FROM fact_loadsmart;
 ---


,total_book_price
0,7106841.76
